In [3]:
import pandas as pd
import numpy as np

## Introduction
This notebook is made to check and redo preprocessing of the MLM-SOS-ALL Project data.
As input the __input_data.csv__ supplied by MS was taken
Some important points regarding the transformations I made:
1. I haven't transformed the encoded variables, since for the downstream analysis we either way encode them again, so it is just unnecessary extra step
2. Whenever it was really left blank or indicated as NA I left the value as NA without putting 0s or anything else. Personally, if it's a NA I think it doesn't mean that it's a negative category, but simply that the value is absent.
3. Instead of putting NA instead of 0s prior to the log transformation of blood counts, I added a pseudocount of 0.1 to all the values in this section. (this step should be thought through)
4. I didn't convert skin prick test values to a compressed category SP_any, I think it is interesting to see what reaction exactly contributes to the outcome.
5. I have dropped Swiss children from this file since they are not used anywhere (yet), they don't have many variables from the core database which further might introduce confusion and bugs.

In [4]:
### specify the path
path_in = "~\\Documents\\Projects\\sosall\\raw-data\\preprocessing\\"
path_out = "~\\Documents\\Projects\\sosall\\results\\preprocessing\\"

## 1. Dictionary creation and renaming
Columns in the raw data file have unstandartized naming, white spaces and other artifacts
Initially, the dictionary containing mapping between old and new names was taken from MS analysis notebook
__TODO__
- Run the analysis with the proper dictionary (using the one I made myself is not a problem, this is done just for a sanity check)

In [5]:
### read-in the data file
data = pd.read_csv(path_in + "input_data.csv")
colnames = data.columns.tolist()
colnames = [i.strip() for i in colnames]  # remove whitespaces in column headers
data.columns = colnames
data = data.apply(lambda x: x.str.strip() if x.dtype == "object" else x)  # remove whitespaces in columns of type object
data.drop(labels = "Location_1", axis = 1, inplace = True)
data.head(8)

,PID,SOSALL_HIV_exposure,PID_type,Diagnosis,Location,Q1=Hospital Name,Q2=Study Site,Q3=Enrolment Date,Q4=Date of Birth,Q5=Age at Enrolment,...,EDEMA/PAPULATION,OOZING/CRUSTING,EXCORIATION,LICHENIFICATION,DRYNESS,Stool Katokatz,PID_2,Subject Code,Group,Country
0,CA-001-LS,Unexposed,1.0,AD,Urban,Red Cross,1.0,02.12.2014,04.12.2012,24.0,...,1.0,0.0,3.0,3.0,2.0,negative,CA-001-LS,NaN,NaN,South Africa
1,CA-002-YF,Unexposed,1.0,AD,Urban,Red Cross,1.0,26.01.2015,25.11.2013,14.0,...,1.0,0.0,2.0,3.0,2.0,No stool,CA-002-YF,NaN,NaN,South Africa
2,CA-003-US,Unexposed,1.0,AD,Urban,Red Cross,1.0,27.01.2015,12.06.2013,20.0,...,NaN,NaN,NaN,NaN,NaN,No stool,CA-003-US,NaN,NaN,South Africa
3,CA-004-NK,Unexposed,1.0,AD,Urban,Red Cross,1.0,28.01.2015,07.12.2013,14.0,...,3.0,1.0,3.0,3.0,3.0,No stool,CA-004-NK,NaN,NaN,South Africa
4,CA-005-BD,Unexposed,1.0,AD,Urban,Red Cross,1.0,29.01.2015,04.09.2013,17.0,...,0.0,0.0,3.0,2.0,3.0,negative,CA-005-BD,NaN,NaN,South Africa
5,CA-006-ON,Unexposed,1.0,AD,Urban,Red Cross,1.0,25.03.2015,15.04.2012,35.0,...,0.0,0.0,3.0,3.0,3.0,negative,CA-006-ON,NaN,NaN,South Africa
6,CA-007-NK,Unexposed,1.0,AD,Urban,Red Cross,1.0,25.03.2015,14.09.2012,30.0,...,0.0,0.0,1.0,2.0,3.0,negative,CA-007-NK,NaN,NaN,South Africa
7,CA-008-HN,Unexposed,1.0,AD,Urban,Red Cross,1.0,31.03.2015,05.06.2012,34.0,...,0.0,0.0,2.0,3.0,3.0,negative,CA-008-HN,NaN,NaN,South Africa


In [6]:
### read the dictionary file
dictionary = pd.read_csv(path_in + "dictionary.csv", sep = ";")
dictionary.columns = ['index', 'working_name']
dictionary.sort_values(by = 'index', inplace = True)
dictionary.reset_index(inplace = True, drop = True)
dictionary

,index,working_name
0,0,PID
1,1,HIV_exp
2,2,PID_type
3,3,diagnosis
4,4,location
...,...,...
242,242,stool_katokatz
243,243,PID_2
244,244,subject_code
245,245,group


In [7]:
### extract the unmodified names of the columns to check if the order of columns in the dictionary corresponds to the order in the raw data
original_names = {"index": [],
                  "original_name": []}

for index, name in enumerate(data.columns):
    original_names['index'].append(index)
    original_names['original_name'].append(name)

original_names = pd.DataFrame(original_names)
original_names.head(100)

,index,original_name
0,0,PID
1,1,SOSALL_HIV_exposure
2,2,PID_type
3,3,Diagnosis
4,4,Location
...,...,...
95,95,Q21.7.3
96,96,Q21.8.1
97,97,Q21.8.2
98,98,Q21.8.3


In [8]:
### map the modified names to the unmodified ones
dictionary = pd.concat([original_names, dictionary], axis = 1)
dictionary

,index,original_name,index,working_name
0,0,PID,0,PID
1,1,SOSALL_HIV_exposure,1,HIV_exp
2,2,PID_type,2,PID_type
3,3,Diagnosis,3,diagnosis
4,4,Location,4,location
...,...,...,...,...
242,242,Stool Katokatz,242,stool_katokatz
243,243,PID_2,243,PID_2
244,244,Subject Code,244,subject_code
245,245,Group,245,group


In [9]:
### make a copy to compare with the original dataframe
data_renamed = data.copy()
data_renamed.columns = dictionary['working_name']
data_renamed.reset_index(drop = True, inplace = True)

### save the dataframe with renamed columns, but without modified features
data_renamed.to_csv(path_out  + "DataRenamed.csv")

In [10]:
data_renamed

working_name,PID,HIV_exp,PID_type,diagnosis,location,hospital_name,study_site,enrolment_date,birthdate,enrolement_age,...,edema,oozing,excoriation,lichenification,dryness,stool_katokatz,PID_2,subject_code,group,country
0,CA-001-LS,Unexposed,1.0,AD,Urban,Red Cross,1.0,02.12.2014,04.12.2012,24.0,...,1.0,0.0,3.0,3.0,2.0,negative,CA-001-LS,NaN,NaN,South Africa
1,CA-002-YF,Unexposed,1.0,AD,Urban,Red Cross,1.0,26.01.2015,25.11.2013,14.0,...,1.0,0.0,2.0,3.0,2.0,No stool,CA-002-YF,NaN,NaN,South Africa
2,CA-003-US,Unexposed,1.0,AD,Urban,Red Cross,1.0,27.01.2015,12.06.2013,20.0,...,NaN,NaN,NaN,NaN,NaN,No stool,CA-003-US,NaN,NaN,South Africa
3,CA-004-NK,Unexposed,1.0,AD,Urban,Red Cross,1.0,28.01.2015,07.12.2013,14.0,...,3.0,1.0,3.0,3.0,3.0,No stool,CA-004-NK,NaN,NaN,South Africa
4,CA-005-BD,Unexposed,1.0,AD,Urban,Red Cross,1.0,29.01.2015,04.09.2013,17.0,...,0.0,0.0,3.0,2.0,3.0,negative,CA-005-BD,NaN,NaN,South Africa
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
240,40010056,NaN,NaN,NaN,NaN,NaN,NaN,19.03.2018,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Switzerland
241,40010052,NaN,NaN,NaN,NaN,NaN,NaN,08.02.2018,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Switzerland
242,40010057,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Switzerland
243,40010054,NaN,NaN,NaN,NaN,NaN,NaN,05.03.2018,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Switzerland


## 2. Generate column report from unmodified data
This is done as a sanity check to compare against the engineered dataframe to spot if something went wrong.
The report dataframe contains following columns:
- *variable* - variable name
- *value_counts* - dictionary containing value and it's frequency for every column
- *uniqe_values* - total number of uniqye values in the column
- *nan_counts* - number of missing values in a column
- *nan_percentage* - percentage of missing values in a column

In [11]:
### make a table with feature value counts, number of unique values and missing value counts
counts = [] # list with value counts for every column
feature_name = [] # list of feature names
na_count = [] # list with na counts in every column
perc_nan = [] # list with percentages of nas in every column
length = [] # list of amount of unique values in every column
for column in data_renamed.columns:
    ### calculate the value counts in each column
    val_counts = data_renamed[column].value_counts().to_dict()
    ### calculate the nan percentage in every column
    nan_sum = data_renamed[column].isna().sum()
    percent_missing = data_renamed[column].isnull().sum() * 100 / len(data_renamed[column])
    ### calculate the number of unique values
    length.append(len(data_renamed[column].unique()))
    ### append the values to list to construct the dataframe
    counts.append(val_counts)
    feature_name.append(column)
    na_count.append(nan_sum)
    perc_nan.append(percent_missing)

### construct dataframe
### for some reason counts NaN as a uniqe value but doesn't show it in the dictionary
df_report = pd.DataFrame(list(zip(feature_name, counts, length, na_count, perc_nan)),
             columns = ['variable','value_counts','unique_values','nan_counts','nan_percentage'])
### save it to the report files
df_report.to_csv(path_out + 'feature_frequencies_report.tsv', sep = '\t')

In [12]:
df_report

,variable,value_counts,unique_values,nan_counts,nan_percentage
0,PID,"{'40010031': 2, 'CA-001-LS': 1, 'CO-102-LP': 1...",244,0,0.000000
1,HIV_exp,"{'unknown': 102, 'Unexposed': 91, 'Exposed': 24}",4,28,11.428571
2,PID_type,"{1.0: 118, 0.0: 99}",3,28,11.428571
3,diagnosis,"{'AD': 116, 'HC': 101}",3,28,11.428571
4,location,"{'Rural': 112, 'Urban': 105}",3,28,11.428571
...,...,...,...,...,...
242,stool_katokatz,"{'negative': 154, 'No stool': 56, 'positive': 7}",4,28,11.428571
243,PID_2,"{'CA-001-LS': 1, 'CO-035-SM': 1, 'CO-024-KM': ...",218,28,11.428571
244,subject_code,"{'CA010EB': 1, 'CO043ZS': 1, 'CO035SM': 1, 'CO...",160,86,35.102041
245,group,"{'CONTROL': 81, 'CASE': 78}",3,86,35.102041


## 3. Feature Enginnering

### 3.1 Removal of unnecessary features

In [13]:
### remove Swiss Children
### Swiss children have incomplete data
### also they are not used in the analysis
data_renamed = data_renamed.loc[data_renamed.country == 'South Africa',]

In [14]:
### remove columns with all the values NaNs
data_renamed.dropna(axis='columns', how='all')

working_name,PID,HIV_exp,PID_type,diagnosis,location,hospital_name,study_site,enrolment_date,birthdate,enrolement_age,...,edema,oozing,excoriation,lichenification,dryness,stool_katokatz,PID_2,subject_code,group,country
0,CA-001-LS,Unexposed,1.0,AD,Urban,Red Cross,1.0,02.12.2014,04.12.2012,24.0,...,1.0,0.0,3.0,3.0,2.0,negative,CA-001-LS,NaN,NaN,South Africa
1,CA-002-YF,Unexposed,1.0,AD,Urban,Red Cross,1.0,26.01.2015,25.11.2013,14.0,...,1.0,0.0,2.0,3.0,2.0,No stool,CA-002-YF,NaN,NaN,South Africa
2,CA-003-US,Unexposed,1.0,AD,Urban,Red Cross,1.0,27.01.2015,12.06.2013,20.0,...,NaN,NaN,NaN,NaN,NaN,No stool,CA-003-US,NaN,NaN,South Africa
3,CA-004-NK,Unexposed,1.0,AD,Urban,Red Cross,1.0,28.01.2015,07.12.2013,14.0,...,3.0,1.0,3.0,3.0,3.0,No stool,CA-004-NK,NaN,NaN,South Africa
4,CA-005-BD,Unexposed,1.0,AD,Urban,Red Cross,1.0,29.01.2015,04.09.2013,17.0,...,0.0,0.0,3.0,2.0,3.0,negative,CA-005-BD,NaN,NaN,South Africa
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,CO-149-LM,unknown,0.0,HC,Rural,Gqubeni,0.0,14.09.2015,04.08.2014,13.0,...,NaN,NaN,NaN,NaN,NaN,No stool,CO-149-LM,CO149LM,CONTROL,South Africa
213,CO-150-LN,unknown,0.0,HC,Rural,Gqubeni,0.0,14.09.2015,15.07.2013,26.0,...,NaN,NaN,NaN,NaN,NaN,No stool,CO-150-LN,NaN,NaN,South Africa
214,CO-151-AN,unknown,0.0,HC,Rural,Gqubeni,0.0,14.09.2015,07.10.2013,23.0,...,NaN,NaN,NaN,NaN,NaN,No stool,CO-151-AN,NaN,NaN,South Africa
215,CO-152-AF,unknown,0.0,HC,Rural,Gqubeni,0.0,14.09.2015,06.07.2013,26.0,...,NaN,NaN,NaN,NaN,NaN,No stool,CO-152-AF,NaN,NaN,South Africa


In [15]:
### sort the columns alphabetically to spot the duplicates
data_renamed.reindex(sorted(data_renamed.columns), axis=1)

working_name,HIV_exp,HIV_rapidELISA,HIV_testresult,PID,PID_1,PID_2,PID_type,am_abd_girth,am_height,am_skinfold,...,study_site,subject_code,sunlight_exp_summer,sunlight_exp_winter,unpasteurizedmilk_exposure,vaccination_status,weight,wheat_exposure,wheat_exposure_age_first,wheat_exposure_regularity
0,Unexposed,1,negative,CA-001-LS,CA-001-LS,CA-001-LS,1.0,47,85.0,10,...,1.0,NaN,7,5,0.0,1.0,12.2,1.0,6.0,1.0
1,Unexposed,1,negative,CA-002-YF,CA-002-YF,CA-002-YF,1.0,43,73.0,6,...,1.0,NaN,0.92,0,0.0,1.0,8.7,1.0,11.0,1.0
2,Unexposed,1,negative,CA-003-US,CA-003-US,CA-003-US,1.0,44,74.0,9,...,1.0,NaN,1.28,0,0.0,1.0,9.7,1.0,17.0,1.0
3,Unexposed,1,negative,CA-004-NK,CA-004-NK,CA-004-NK,1.0,42,80.0,9,...,1.0,NaN,0.35,0,0.0,1.0,11.4,1.0,5.0,0.0
4,Unexposed,1,negative,CA-005-BD,CA-005-BD,CA-005-BD,1.0,44,73.0,8,...,1.0,NaN,0.78,0.5,0.0,1.0,8.5,1.0,4.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,unknown,1,NaN,CO-149-LM,CO-149-LM,CO-149-LM,0.0,45,76.0,11,...,0.0,CO149LM,2,3,0.0,1.0,11.0,1.0,12.0,1.0
213,unknown,1,NaN,CO-150-LN,CO-150-LN,CO-150-LN,0.0,44,87.0,12,...,0.0,NaN,5,1,0.0,1.0,13.5,1.0,24.0,1.0
214,unknown,1,NaN,CO-151-AN,CO-151-AN,CO-151-AN,0.0,44,84.0,10,...,0.0,NaN,1,2,0.0,1.0,12.0,1.0,19.0,1.0
215,unknown,1,NaN,CO-152-AF,CO-152-AF,CO-152-AF,0.0,49,87.0,10,...,0.0,NaN,9,2,0.0,1.0,12.8,1.0,18.0,1.0


In [16]:
### drop the HIV and PID columns except PID
data_renamed = data_renamed.drop(data_renamed.filter(regex = 'HIV').columns, axis = 1)
data_renamed = data_renamed.drop(labels = ['PID_1', 'PID_2', 'PID_type'], axis = 1)
data_renamed

working_name,PID,diagnosis,location,hospital_name,study_site,enrolment_date,birthdate,enrolement_age,gender,weight,...,erythema,edema,oozing,excoriation,lichenification,dryness,stool_katokatz,subject_code,group,country
0,CA-001-LS,AD,Urban,Red Cross,1.0,02.12.2014,04.12.2012,24.0,2.0,12.2,...,0.0,1.0,0.0,3.0,3.0,2.0,negative,NaN,NaN,South Africa
1,CA-002-YF,AD,Urban,Red Cross,1.0,26.01.2015,25.11.2013,14.0,1.0,8.7,...,1.0,1.0,0.0,2.0,3.0,2.0,No stool,NaN,NaN,South Africa
2,CA-003-US,AD,Urban,Red Cross,1.0,27.01.2015,12.06.2013,20.0,2.0,9.7,...,NaN,NaN,NaN,NaN,NaN,NaN,No stool,NaN,NaN,South Africa
3,CA-004-NK,AD,Urban,Red Cross,1.0,28.01.2015,07.12.2013,14.0,1.0,11.4,...,1.0,3.0,1.0,3.0,3.0,3.0,No stool,NaN,NaN,South Africa
4,CA-005-BD,AD,Urban,Red Cross,1.0,29.01.2015,04.09.2013,17.0,2.0,8.5,...,0.0,0.0,0.0,3.0,2.0,3.0,negative,NaN,NaN,South Africa
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,CO-149-LM,HC,Rural,Gqubeni,0.0,14.09.2015,04.08.2014,13.0,2.0,11.0,...,NaN,NaN,NaN,NaN,NaN,NaN,No stool,CO149LM,CONTROL,South Africa
213,CO-150-LN,HC,Rural,Gqubeni,0.0,14.09.2015,15.07.2013,26.0,1.0,13.5,...,NaN,NaN,NaN,NaN,NaN,NaN,No stool,NaN,NaN,South Africa
214,CO-151-AN,HC,Rural,Gqubeni,0.0,14.09.2015,07.10.2013,23.0,1.0,12.0,...,NaN,NaN,NaN,NaN,NaN,NaN,No stool,NaN,NaN,South Africa
215,CO-152-AF,HC,Rural,Gqubeni,0.0,14.09.2015,06.07.2013,26.0,1.0,12.8,...,NaN,NaN,NaN,NaN,NaN,NaN,No stool,NaN,NaN,South Africa


In [17]:
### drop the columns that were specified in the Marco's pipeline
cols = ['study_site', 'height', 'weight', 'comments', 'group', 'childhood_inf_measles',
        'childhood_inf_rubella', 'childhood_inf_hepatitis', 'childhood_inf_specify',
        'medication_homenebuliser', 'medication_controllers', 'medication_adrenalin', 'medication_antihistamines_which','parental_job_woman',
        'parental_job_man', 'erythema', 'edema', 'oozing', 'excoriation', 'lichenification', 'dryness','asthma_age', 'asthma_whodiagnosed', 'hayfever_age', 'hayfever_whodiagnosed',
        'medication_anyother', 'medicalhistory_comorbidities_which', 'cigarettesmoke_mother_howmany',
        'cigarettesmoke_father_howmany', 'itch', 'onset_before_2', 'hx_flexural_ad', 'hx_dry_skin',
        'personal_1degree_atopy', 'flexural_ad_vis', 'cigarettesmoke_father', 'cigarettesmoke_mother',
        'cigarettesmoke_father_howmany', 'cigarettesmoke_father_howmany', 'medication_relievers', 'ethnicity', 'pe_ar', 'pe_asthma',
        'enrolment_date','birthdate', 'hospital_name', 'pe_ar', 'pe_asthma', 'medication_relievers', 'unpasteurizedmilk_exposure' ]  # unpasterizedmilk is not present

In [18]:
### drop specified columns
data_renamed = data_renamed.drop(labels = cols, axis = 1)
data_renamed

working_name,PID,diagnosis,location,enrolement_age,gender,immun_bcg_birth,immun_opv_0_birth,immun_opv_1_6w,immun_rv_1_6w,immun_dtapipv_hib_1_6w,...,blood_count_plts,blood_count_neutrophils,blood_count_monocytes,blood_count_lymphocytes,blood_count_eosinophils,scorad,bsa,stool_katokatz,subject_code,country
0,CA-001-LS,AD,Urban,24.0,2.0,1.0,1.0,1.0,1.0,1.0,...,476,3.32,0.75,4.28,2.36,33.1,8.0,negative,NaN,South Africa
1,CA-002-YF,AD,Urban,14.0,1.0,1.0,1.0,1.0,1.0,1.0,...,483,3.46,1.24,6.92,0.74,34.3,14.0,No stool,NaN,South Africa
2,CA-003-US,AD,Urban,20.0,2.0,1.0,1.0,1.0,1.0,1.0,...,123,2.21,0.39,4.50,0.71,NR,NaN,No stool,NaN,South Africa
3,CA-004-NK,AD,Urban,14.0,1.0,1.0,1.0,1.0,1.0,1.0,...,291,2.57,0.77,7.06,2.18,60,54.0,No stool,NaN,South Africa
4,CA-005-BD,AD,Urban,17.0,2.0,1.0,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,37.5,31.0,negative,NaN,South Africa
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,CO-149-LM,HC,Rural,13.0,2.0,1.0,1.0,1.0,1.0,1.0,...,343,1.81,1.08,5.42,0.76,NaN,NaN,No stool,CO149LM,South Africa
213,CO-150-LN,HC,Rural,26.0,1.0,1.0,1.0,1.0,1.0,1.0,...,222,1.74,0.46,3.96,0.64,NaN,NaN,No stool,NaN,South Africa
214,CO-151-AN,HC,Rural,23.0,1.0,1.0,1.0,1.0,1.0,1.0,...,342,2.24,1.26,6.84,0.34,NaN,NaN,No stool,NaN,South Africa
215,CO-152-AF,HC,Rural,26.0,1.0,1.0,1.0,1.0,1.0,1.0,...,257,3.49,0.96,5.02,0.72,NaN,NaN,No stool,NaN,South Africa


In [19]:
### remove columns containing variables of following groups:
# amasi
# cowmilk
# probiotic
# soyamilk
# soyaproducts
# homelanguage

data_renamed = data_renamed.drop(data_renamed.filter(regex ='amasi_|cowmilk_|probiotic_|soyamilk_|soyaproducts_|homelanguage_').columns, axis = 1)
data_renamed = data_renamed.drop(labels = 'subject_code', axis = 1) # forgot to drop subject doe along the way
data_renamed

working_name,PID,diagnosis,location,enrolement_age,gender,immun_bcg_birth,immun_opv_0_birth,immun_opv_1_6w,immun_rv_1_6w,immun_dtapipv_hib_1_6w,...,blood_count_hb,blood_count_plts,blood_count_neutrophils,blood_count_monocytes,blood_count_lymphocytes,blood_count_eosinophils,scorad,bsa,stool_katokatz,country
0,CA-001-LS,AD,Urban,24.0,2.0,1.0,1.0,1.0,1.0,1.0,...,11.4,476,3.32,0.75,4.28,2.36,33.1,8.0,negative,South Africa
1,CA-002-YF,AD,Urban,14.0,1.0,1.0,1.0,1.0,1.0,1.0,...,11.3,483,3.46,1.24,6.92,0.74,34.3,14.0,No stool,South Africa
2,CA-003-US,AD,Urban,20.0,2.0,1.0,1.0,1.0,1.0,1.0,...,10.8,123,2.21,0.39,4.50,0.71,NR,NaN,No stool,South Africa
3,CA-004-NK,AD,Urban,14.0,1.0,1.0,1.0,1.0,1.0,1.0,...,13.0,291,2.57,0.77,7.06,2.18,60,54.0,No stool,South Africa
4,CA-005-BD,AD,Urban,17.0,2.0,1.0,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,37.5,31.0,negative,South Africa
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,CO-149-LM,HC,Rural,13.0,2.0,1.0,1.0,1.0,1.0,1.0,...,10.0,343,1.81,1.08,5.42,0.76,NaN,NaN,No stool,South Africa
213,CO-150-LN,HC,Rural,26.0,1.0,1.0,1.0,1.0,1.0,1.0,...,10.4,222,1.74,0.46,3.96,0.64,NaN,NaN,No stool,South Africa
214,CO-151-AN,HC,Rural,23.0,1.0,1.0,1.0,1.0,1.0,1.0,...,11.8,342,2.24,1.26,6.84,0.34,NaN,NaN,No stool,South Africa
215,CO-152-AF,HC,Rural,26.0,1.0,1.0,1.0,1.0,1.0,1.0,...,10.5,257,3.49,0.96,5.02,0.72,NaN,NaN,No stool,South Africa


### 3.2 __Foodreaction__ group of features (foodreaction_)

In [20]:
### collapse foodreaction columns into foodreaction_any
### according to the Maroc's script
### display columns with foodreaction variables
for columns in data_renamed.loc[:,data_renamed.columns.str.contains('foodreaction')].columns:
    print(columns)

foodreaction_peanuts_ever
foodreaction_peanuts_which
foodreaction_peanuts_age
foodreaction_peanuts_when
foodreaction_peanuts_confirmed
foodreaction_othernuts_ever
foodreaction_othernuts_which
foodreaction_othernuts_age
foodreaction_othernuts_when
foodreaction_othernuts_confirmed
foodreaction_diaryproducts_ever
foodreaction_diaryproducts_which
foodreaction_diaryproducts_age
foodreaction_diaryproducts_when
foodreaction_diaryproducts_confirmed
foodreaction_soya_ever
foodreaction_soya_which
foodreaction_soya_age
foodreaction_soya_when
foodreaction_soya_confirmed
foodreaction_henegg_ever
foodreaction_henegg_which
foodreaction_henegg_age
foodreaction_henegg_when
foodreaction_henegg_confirmed
foodreaction_wheat_ever
foodreaction_wheat_which
foodreaction_wheat_age
foodreaction_wheat_when
foodreaction_wheat_confirmed
foodreaction_fish_ever
foodreaction_fish_which
foodreaction_fish_age
foodreaction_fish_when
foodreaction_fish_confirmed


In [21]:
### check if regex works as intended
for columns in data_renamed.loc[:,data_renamed.columns.str.contains('foodreaction_(.*?)_which|foodreaction_(.*?)_age|foodreaction_(.*?)_when|foodreaction_(.*?)_confirmed', regex = True)].columns:
    print(columns)

foodreaction_peanuts_which
foodreaction_peanuts_age
foodreaction_peanuts_when
foodreaction_peanuts_confirmed
foodreaction_othernuts_which
foodreaction_othernuts_age
foodreaction_othernuts_when
foodreaction_othernuts_confirmed
foodreaction_diaryproducts_which
foodreaction_diaryproducts_age
foodreaction_diaryproducts_when
foodreaction_diaryproducts_confirmed
foodreaction_soya_which
foodreaction_soya_age
foodreaction_soya_when
foodreaction_soya_confirmed
foodreaction_henegg_which
foodreaction_henegg_age
foodreaction_henegg_when
foodreaction_henegg_confirmed
foodreaction_wheat_which
foodreaction_wheat_age
foodreaction_wheat_when
foodreaction_wheat_confirmed
foodreaction_fish_which
foodreaction_fish_age
foodreaction_fish_when
foodreaction_fish_confirmed


C:\Temp\ipykernel_23352\3643446379.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  for columns in data_renamed.loc[:,data_renamed.columns.str.contains('foodreaction_(.*?)_which|foodreaction_(.*?)_age|foodreaction_(.*?)_when|foodreaction_(.*?)_confirmed', regex = True)].columns:


In [22]:
### drop the columns containing foodrection_which, foodreaction_age, foodreaction_when, foodreaction_confirmed
data_renamed = data_renamed.drop(data_renamed.filter(regex ='foodreaction_(.*?)_which|foodreaction_(.*?)_age|foodreaction_(.*?)_when|foodreaction_(.*?)_confirmed').columns, axis = 1)
data_renamed

working_name,PID,diagnosis,location,enrolement_age,gender,immun_bcg_birth,immun_opv_0_birth,immun_opv_1_6w,immun_rv_1_6w,immun_dtapipv_hib_1_6w,...,blood_count_hb,blood_count_plts,blood_count_neutrophils,blood_count_monocytes,blood_count_lymphocytes,blood_count_eosinophils,scorad,bsa,stool_katokatz,country
0,CA-001-LS,AD,Urban,24.0,2.0,1.0,1.0,1.0,1.0,1.0,...,11.4,476,3.32,0.75,4.28,2.36,33.1,8.0,negative,South Africa
1,CA-002-YF,AD,Urban,14.0,1.0,1.0,1.0,1.0,1.0,1.0,...,11.3,483,3.46,1.24,6.92,0.74,34.3,14.0,No stool,South Africa
2,CA-003-US,AD,Urban,20.0,2.0,1.0,1.0,1.0,1.0,1.0,...,10.8,123,2.21,0.39,4.50,0.71,NR,NaN,No stool,South Africa
3,CA-004-NK,AD,Urban,14.0,1.0,1.0,1.0,1.0,1.0,1.0,...,13.0,291,2.57,0.77,7.06,2.18,60,54.0,No stool,South Africa
4,CA-005-BD,AD,Urban,17.0,2.0,1.0,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,37.5,31.0,negative,South Africa
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,CO-149-LM,HC,Rural,13.0,2.0,1.0,1.0,1.0,1.0,1.0,...,10.0,343,1.81,1.08,5.42,0.76,NaN,NaN,No stool,South Africa
213,CO-150-LN,HC,Rural,26.0,1.0,1.0,1.0,1.0,1.0,1.0,...,10.4,222,1.74,0.46,3.96,0.64,NaN,NaN,No stool,South Africa
214,CO-151-AN,HC,Rural,23.0,1.0,1.0,1.0,1.0,1.0,1.0,...,11.8,342,2.24,1.26,6.84,0.34,NaN,NaN,No stool,South Africa
215,CO-152-AF,HC,Rural,26.0,1.0,1.0,1.0,1.0,1.0,1.0,...,10.5,257,3.49,0.96,5.02,0.72,NaN,NaN,No stool,South Africa


In [23]:
### combine foodreaction_(.*?)_ever columns into one foodreaction_ever
data_renamed['foodreaction_any'] = data_renamed.loc[:,data_renamed.columns.str.contains('foodreaction_(.*?)_ever', regex = True)].max(axis = 1)
data_renamed

C:\Temp\ipykernel_23352\1580543485.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  data_renamed['foodreaction_any'] = data_renamed.loc[:,data_renamed.columns.str.contains('foodreaction_(.*?)_ever', regex = True)].max(axis = 1)
C:\Temp\ipykernel_23352\1580543485.py:2: FutureWarning: Dropping of nuisance columns in DataFrame reductions (with 'numeric_only=None') is deprecated; in a future version this will raise TypeError.  Select only valid columns before calling the reduction.
  data_renamed['foodreaction_any'] = data_renamed.loc[:,data_renamed.columns.str.contains('foodreaction_(.*?)_ever', regex = True)].max(axis = 1)


working_name,PID,diagnosis,location,enrolement_age,gender,immun_bcg_birth,immun_opv_0_birth,immun_opv_1_6w,immun_rv_1_6w,immun_dtapipv_hib_1_6w,...,blood_count_plts,blood_count_neutrophils,blood_count_monocytes,blood_count_lymphocytes,blood_count_eosinophils,scorad,bsa,stool_katokatz,country,foodreaction_any
0,CA-001-LS,AD,Urban,24.0,2.0,1.0,1.0,1.0,1.0,1.0,...,476,3.32,0.75,4.28,2.36,33.1,8.0,negative,South Africa,0.0
1,CA-002-YF,AD,Urban,14.0,1.0,1.0,1.0,1.0,1.0,1.0,...,483,3.46,1.24,6.92,0.74,34.3,14.0,No stool,South Africa,0.0
2,CA-003-US,AD,Urban,20.0,2.0,1.0,1.0,1.0,1.0,1.0,...,123,2.21,0.39,4.50,0.71,NR,NaN,No stool,South Africa,0.0
3,CA-004-NK,AD,Urban,14.0,1.0,1.0,1.0,1.0,1.0,1.0,...,291,2.57,0.77,7.06,2.18,60,54.0,No stool,South Africa,1.0
4,CA-005-BD,AD,Urban,17.0,2.0,1.0,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,37.5,31.0,negative,South Africa,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,CO-149-LM,HC,Rural,13.0,2.0,1.0,1.0,1.0,1.0,1.0,...,343,1.81,1.08,5.42,0.76,NaN,NaN,No stool,South Africa,0.0
213,CO-150-LN,HC,Rural,26.0,1.0,1.0,1.0,1.0,1.0,1.0,...,222,1.74,0.46,3.96,0.64,NaN,NaN,No stool,South Africa,0.0
214,CO-151-AN,HC,Rural,23.0,1.0,1.0,1.0,1.0,1.0,1.0,...,342,2.24,1.26,6.84,0.34,NaN,NaN,No stool,South Africa,0.0
215,CO-152-AF,HC,Rural,26.0,1.0,1.0,1.0,1.0,1.0,1.0,...,257,3.49,0.96,5.02,0.72,NaN,NaN,No stool,South Africa,0.0


In [24]:
### remove foodreaction_(.*?)_ever variables
data_renamed = data_renamed.drop(data_renamed.filter(regex ='foodreaction_(.*?)_ever').columns, axis = 1)
data_renamed.columns

Index(['PID', 'diagnosis', 'location', 'enrolement_age', 'gender',
       'immun_bcg_birth', 'immun_opv_0_birth', 'immun_opv_1_6w',
       'immun_rv_1_6w', 'immun_dtapipv_hib_1_6w',
       ...
       'blood_count_plts', 'blood_count_neutrophils', 'blood_count_monocytes',
       'blood_count_lymphocytes', 'blood_count_eosinophils', 'scorad', 'bsa',
       'stool_katokatz', 'country', 'foodreaction_any'],
      dtype='object', name='working_name', length=131)

### 3.3 Skin Prick test features (sp_)

In [25]:
### not touching any SP values because they are important for the downstream analysis
### however negative and positive controls should be removed
for columns in data_renamed.loc[:,data_renamed.columns.str.contains('sp_', regex = True)].columns:
    print(columns)

sp_neg_control
sp_egg_white
sp_cowmilk
sp_soy
sp_wheet
sp_fish
sp_peanut
sp_hazelnut
sp_positive_control
sp_fresh_peanut
sp_fresh_egg
sp_fresh_cowmilk


In [26]:
data_renamed.drop(labels = ['sp_neg_control', 'sp_positive_control'], axis = 1, inplace = True)
print(data_renamed.loc[:,data_renamed.columns.str.contains('sp_')].columns)

Index(['sp_egg_white', 'sp_cowmilk', 'sp_soy', 'sp_wheet', 'sp_fish',
       'sp_peanut', 'sp_hazelnut', 'sp_fresh_peanut', 'sp_fresh_egg',
       'sp_fresh_cowmilk'],
      dtype='object', name='working_name')


### 3.4 Immunization features (immun_)

In [27]:
### doing immune_number feature combination
for columns in data_renamed.loc[:,data_renamed.columns.str.contains('immun_', regex = True)].columns:
    print(columns)
print('The total number of immun_ features: {}'.format(len(data_renamed.loc[:,data_renamed.columns.str.contains('immun_', regex = True)].columns)))

immun_bcg_birth
immun_opv_0_birth
immun_opv_1_6w
immun_rv_1_6w
immun_dtapipv_hib_1_6w
immun_hepb_1_6w
immun_pcv713_1_6w
immun_dtapipv_hib_2_10w
immun_hepb_2_10w
immun_rv_2_14w
immun_dtapipv_hib_3_14w
immun_hepb3_14w
immun_pcv713_2_14w
immun_measles_9m
immun_pcv713_3_9m
immun_dtapipv_hib_4_18m
immun_measales_18m
The total number of immun_ features: 17


In [28]:
data_renamed['immun_number'] = data_renamed.loc[:,data_renamed.columns.str.contains('immun_', regex = True)].sum(axis = 1, skipna = False)
data_renamed.immun_number

0      17.0
1      15.0
2      17.0
3      15.0
4      17.0
       ... 
212    15.0
213    17.0
214    17.0
215    17.0
216    15.0
Name: immun_number, Length: 217, dtype: float64

In [29]:
### drop other immun_ columns except immun_number
### regex is too hard with this one, just passing the columns names instead
data_renamed.drop(labels = ['immun_bcg_birth',
                            'immun_opv_0_birth',
                            'immun_opv_1_6w',
                            'immun_rv_1_6w',
                            'immun_dtapipv_hib_1_6w',
                            'immun_hepb_1_6w',
                            'immun_pcv713_1_6w',
                            'immun_dtapipv_hib_2_10w',
                            'immun_hepb_2_10w',
                            'immun_rv_2_14w',
                            'immun_dtapipv_hib_3_14w',
                            'immun_hepb3_14w',
                            'immun_pcv713_2_14w',
                            'immun_measles_9m',
                            'immun_pcv713_3_9m',
                            'immun_dtapipv_hib_4_18m',
                            'immun_measales_18m'], axis = 1, inplace = True)

In [30]:
data_renamed

working_name,PID,diagnosis,location,enrolement_age,gender,vaccination_status,paracetamol_exposure,paracetamol_exposure_age,paracetamol_exposure_time,childhood_inf_mumps,...,blood_count_neutrophils,blood_count_monocytes,blood_count_lymphocytes,blood_count_eosinophils,scorad,bsa,stool_katokatz,country,foodreaction_any,immun_number
0,CA-001-LS,AD,Urban,24.0,2.0,1.0,1.0,11.0,1.0,0.0,...,3.32,0.75,4.28,2.36,33.1,8.0,negative,South Africa,0.0,17.0
1,CA-002-YF,AD,Urban,14.0,1.0,1.0,1.0,8.0,1.0,0.0,...,3.46,1.24,6.92,0.74,34.3,14.0,No stool,South Africa,0.0,15.0
2,CA-003-US,AD,Urban,20.0,2.0,1.0,1.0,4.0,3.0,0.0,...,2.21,0.39,4.50,0.71,NR,NaN,No stool,South Africa,0.0,17.0
3,CA-004-NK,AD,Urban,14.0,1.0,1.0,1.0,1.0,1.0,0.0,...,2.57,0.77,7.06,2.18,60,54.0,No stool,South Africa,1.0,15.0
4,CA-005-BD,AD,Urban,17.0,2.0,1.0,1.0,6.0,1.0,0.0,...,NaN,NaN,NaN,NaN,37.5,31.0,negative,South Africa,0.0,17.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,CO-149-LM,HC,Rural,13.0,2.0,1.0,1.0,3.0,1.0,0.0,...,1.81,1.08,5.42,0.76,NaN,NaN,No stool,South Africa,0.0,15.0
213,CO-150-LN,HC,Rural,26.0,1.0,1.0,1.0,NaN,NaN,0.0,...,1.74,0.46,3.96,0.64,NaN,NaN,No stool,South Africa,0.0,17.0
214,CO-151-AN,HC,Rural,23.0,1.0,1.0,1.0,7.0,2.0,0.0,...,2.24,1.26,6.84,0.34,NaN,NaN,No stool,South Africa,0.0,17.0
215,CO-152-AF,HC,Rural,26.0,1.0,1.0,1.0,9.0,2.0,0.0,...,3.49,0.96,5.02,0.72,NaN,NaN,No stool,South Africa,0.0,17.0


In [31]:
### replace ??? with NaNs
data_renamed = data_renamed.replace('???',np.NaN)
data_renamed

working_name,PID,diagnosis,location,enrolement_age,gender,vaccination_status,paracetamol_exposure,paracetamol_exposure_age,paracetamol_exposure_time,childhood_inf_mumps,...,blood_count_neutrophils,blood_count_monocytes,blood_count_lymphocytes,blood_count_eosinophils,scorad,bsa,stool_katokatz,country,foodreaction_any,immun_number
0,CA-001-LS,AD,Urban,24.0,2.0,1.0,1.0,11.0,1.0,0.0,...,3.32,0.75,4.28,2.36,33.1,8.0,negative,South Africa,0.0,17.0
1,CA-002-YF,AD,Urban,14.0,1.0,1.0,1.0,8.0,1.0,0.0,...,3.46,1.24,6.92,0.74,34.3,14.0,No stool,South Africa,0.0,15.0
2,CA-003-US,AD,Urban,20.0,2.0,1.0,1.0,4.0,3.0,0.0,...,2.21,0.39,4.50,0.71,NR,NaN,No stool,South Africa,0.0,17.0
3,CA-004-NK,AD,Urban,14.0,1.0,1.0,1.0,1.0,1.0,0.0,...,2.57,0.77,7.06,2.18,60,54.0,No stool,South Africa,1.0,15.0
4,CA-005-BD,AD,Urban,17.0,2.0,1.0,1.0,6.0,1.0,0.0,...,NaN,NaN,NaN,NaN,37.5,31.0,negative,South Africa,0.0,17.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,CO-149-LM,HC,Rural,13.0,2.0,1.0,1.0,3.0,1.0,0.0,...,1.81,1.08,5.42,0.76,NaN,NaN,No stool,South Africa,0.0,15.0
213,CO-150-LN,HC,Rural,26.0,1.0,1.0,1.0,NaN,NaN,0.0,...,1.74,0.46,3.96,0.64,NaN,NaN,No stool,South Africa,0.0,17.0
214,CO-151-AN,HC,Rural,23.0,1.0,1.0,1.0,7.0,2.0,0.0,...,2.24,1.26,6.84,0.34,NaN,NaN,No stool,South Africa,0.0,17.0
215,CO-152-AF,HC,Rural,26.0,1.0,1.0,1.0,9.0,2.0,0.0,...,3.49,0.96,5.02,0.72,NaN,NaN,No stool,South Africa,0.0,17.0


### 3.5 Family history variables (familyhistory)

In [32]:
### family history
### visualize all familyhistory columns
for columns in data_renamed.loc[:,data_renamed.columns.str.contains('familyhistory_', regex = True)].columns:
    print(columns)
print('The total number of immun_ features: {}'.format(len(data_renamed.loc[:,data_renamed.columns.str.contains('familyhistory_', regex = True)].columns)))

familyhistory_mother
familyhistory_father
familyhistory_sibling1
familyhistory_sibling2
familyhistory_sibling3
familyhistory_sibling4
The total number of immun_ features: 6


In [33]:
data_renamed.loc[:,data_renamed.columns.str.contains('familyhistory_', regex = True)]

working_name,familyhistory_mother,familyhistory_father,familyhistory_sibling1,familyhistory_sibling2,familyhistory_sibling3,familyhistory_sibling4
0,1,1,NaN,NaN,NaN,NaN
1,3,1,1,NaN,NaN,NaN
2,3;4,1,NaN,NaN,NaN,NaN
3,1,4,NaN,NaN,NaN,NaN
4,4,1,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
212,1,1,NaN,NaN,NaN,NaN
213,1,1,NaN,NaN,NaN,NaN
214,1,1,1,NaN,NaN,NaN
215,1,1,NaN,NaN,NaN,NaN


In [34]:
### collapse familyhistory_siblings into one column
columns = ['familyhistory_sibling1', 'familyhistory_sibling2', 'familyhistory_sibling3', 'familyhistory_sibling4']

data_renamed['familyhistory_sibling'] = \
    data_renamed[data_renamed[columns].columns[0:]].apply(
    lambda x: ';'.join(x.dropna().astype(str)),
    axis=1)

data_renamed['familyhistory_sibling'] = data_renamed.familyhistory_sibling.replace(r'^\s*$', np.nan, regex=True)

data_renamed

working_name,PID,diagnosis,location,enrolement_age,gender,vaccination_status,paracetamol_exposure,paracetamol_exposure_age,paracetamol_exposure_time,childhood_inf_mumps,...,blood_count_monocytes,blood_count_lymphocytes,blood_count_eosinophils,scorad,bsa,stool_katokatz,country,foodreaction_any,immun_number,familyhistory_sibling
0,CA-001-LS,AD,Urban,24.0,2.0,1.0,1.0,11.0,1.0,0.0,...,0.75,4.28,2.36,33.1,8.0,negative,South Africa,0.0,17.0,NaN
1,CA-002-YF,AD,Urban,14.0,1.0,1.0,1.0,8.0,1.0,0.0,...,1.24,6.92,0.74,34.3,14.0,No stool,South Africa,0.0,15.0,1
2,CA-003-US,AD,Urban,20.0,2.0,1.0,1.0,4.0,3.0,0.0,...,0.39,4.50,0.71,NR,NaN,No stool,South Africa,0.0,17.0,NaN
3,CA-004-NK,AD,Urban,14.0,1.0,1.0,1.0,1.0,1.0,0.0,...,0.77,7.06,2.18,60,54.0,No stool,South Africa,1.0,15.0,NaN
4,CA-005-BD,AD,Urban,17.0,2.0,1.0,1.0,6.0,1.0,0.0,...,NaN,NaN,NaN,37.5,31.0,negative,South Africa,0.0,17.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,CO-149-LM,HC,Rural,13.0,2.0,1.0,1.0,3.0,1.0,0.0,...,1.08,5.42,0.76,NaN,NaN,No stool,South Africa,0.0,15.0,NaN
213,CO-150-LN,HC,Rural,26.0,1.0,1.0,1.0,NaN,NaN,0.0,...,0.46,3.96,0.64,NaN,NaN,No stool,South Africa,0.0,17.0,NaN
214,CO-151-AN,HC,Rural,23.0,1.0,1.0,1.0,7.0,2.0,0.0,...,1.26,6.84,0.34,NaN,NaN,No stool,South Africa,0.0,17.0,1
215,CO-152-AF,HC,Rural,26.0,1.0,1.0,1.0,9.0,2.0,0.0,...,0.96,5.02,0.72,NaN,NaN,No stool,South Africa,0.0,17.0,NaN


In [35]:
df_sibling = data_renamed["familyhistory_sibling"].fillna('Missing').str.get_dummies(";")

### combine the columns with the same numbering into one
### for some reason it was filled in different way with floats and ints in some cases
df_sibling['1_collapsed'] = df_sibling.loc[:,['1','1.0']].max(axis = 1)
df_sibling['2_collapsed'] = df_sibling.loc[:,['2','2.0']].max(axis = 1)
df_sibling['3_collapsed'] = df_sibling.loc[:,['3','3.0']].max(axis = 1)

df_sibling.drop(labels = ['1','1.0','2','2.0','3','3.0'], axis = 1, inplace = True)
df_sibling.columns = ['familyhistory_sibling_eczema', 'familyhistory_sibling_foodallergy', 'familyhistory_sibling_nan', 'familyhistory_sibling_none', 'familyhistory_sibling_asthma', 'familyhistory_sibling_hayfever']

### assign NaNs to the entire row if the foodreaction_nan has NaNs
idx = df_sibling[df_sibling.familyhistory_sibling_nan == 1].index
df_sibling.loc[idx,:] = np.NaN

### same as previously but for father
df_sibling['familyhistory_sibling_other_allergic_disease'] = df_sibling.loc[:,['familyhistory_sibling_asthma','familyhistory_sibling_foodallergy','familyhistory_sibling_hayfever']].max(axis = 1)
df_sibling.drop(labels = ['familyhistory_sibling_asthma','familyhistory_sibling_foodallergy','familyhistory_sibling_hayfever', 'familyhistory_sibling_nan', 'familyhistory_sibling_none'], axis = 1, inplace = True)
df_sibling.head(50)

,familyhistory_sibling_eczema,familyhistory_sibling_other_allergic_disease
0,NaN,NaN
1,0.0,0.0
2,NaN,NaN
3,NaN,NaN
4,NaN,NaN
5,NaN,NaN
6,NaN,NaN
7,1.0,0.0
8,NaN,NaN
9,NaN,NaN


In [36]:
df_mother = data_renamed["familyhistory_mother"].fillna('Missing').str.get_dummies(";")
df_mother.columns = ['familyhistory_mother_none', 'familyhistory_mother_asthma', 'familyhistory_mother_hayfever', 'familyhistory_mother_eczema', 'familyhistory_mother_foodallergy', 'familyhistory_mother_nan']

### assign NaNs to the entire row if the foodreaction_nan has NaNs
idx = df_mother[df_mother.familyhistory_mother_nan == 1].index
df_mother.loc[idx,:] = np.NaN

### collapse all allergy diseases columns except eczema into one
### then drop unnecessary columns:
### 'familyhistory_mother_asthma','familyhistory_mother_foodallergy','familyhistory_mother_hayfever', 'familyhistory_mother_nan', 'familyhistory_mother_none'
df_mother['familyhistory_mother_other_allergic_disease'] = df_mother.loc[:,['familyhistory_mother_asthma','familyhistory_mother_foodallergy','familyhistory_mother_hayfever']].max(axis = 1)
df_mother.drop(labels = ['familyhistory_mother_asthma','familyhistory_mother_foodallergy','familyhistory_mother_hayfever', 'familyhistory_mother_nan', 'familyhistory_mother_none'], axis = 1, inplace = True)
df_mother.head(50)

,familyhistory_mother_eczema,familyhistory_mother_other_allergic_disease
0,0.0,0.0
1,0.0,1.0
2,1.0,1.0
3,0.0,0.0
4,1.0,0.0
5,0.0,0.0
6,0.0,0.0
7,1.0,0.0
8,0.0,0.0
9,NaN,NaN


In [37]:
df_father = data_renamed["familyhistory_father"].fillna('Missing').str.get_dummies(";")
df_father.columns = ['familyhistory_father_none', 'familyhistory_father_asthma', 'familyhistory_father_hayfever', 'familyhistory_father_eczema', 'familyhistory_father_foodallergy', 'familyhistory_father_nan']

### assign NaNs to the entire row if the foodreaction_nan has NaNs
idx = df_father[df_father.familyhistory_father_nan == 1].index
df_father.loc[idx,:] = np.NaN

### same as previously but for father
df_father['familyhistory_father_other_allergic_disease'] = df_father.loc[:,['familyhistory_father_asthma','familyhistory_father_foodallergy','familyhistory_father_hayfever']].max(axis = 1)
df_father.drop(labels = ['familyhistory_father_asthma','familyhistory_father_foodallergy','familyhistory_father_hayfever', 'familyhistory_father_nan', 'familyhistory_father_none'], axis = 1, inplace = True)
df_father.head(50)

,familyhistory_father_eczema,familyhistory_father_other_allergic_disease
0,0.0,0.0
1,0.0,0.0
2,0.0,0.0
3,1.0,0.0
4,0.0,0.0
5,0.0,0.0
6,0.0,0.0
7,0.0,0.0
8,0.0,0.0
9,NaN,NaN


In [38]:
### create a dataframe comparing columns before and after transformation
df_family_comparison = pd.concat([data_renamed['familyhistory_mother'], df_mother,
           data_renamed['familyhistory_father'], df_father,
           data_renamed.loc[:,data_renamed.columns.str.contains('sibling')], df_sibling],
          axis = 1)
df_family_comparison.to_csv(path_out + 'DataFamilyTransformation.txt', na_rep = 'NaN',sep = '\t')

In [39]:
### drop the columns with familyhistory variables from the final dataframe and add transformed columns
df_family_transformed = pd.concat([df_mother, df_father, df_sibling], axis = 1)
data_renamed.drop(labels = data_renamed.filter(regex ='familyhistory').columns, axis = 1, inplace = True)
data_renamed = pd.concat([data_renamed, df_family_transformed], axis = 1)
data_renamed

,PID,diagnosis,location,enrolement_age,gender,vaccination_status,paracetamol_exposure,paracetamol_exposure_age,paracetamol_exposure_time,childhood_inf_mumps,...,stool_katokatz,country,foodreaction_any,immun_number,familyhistory_mother_eczema,familyhistory_mother_other_allergic_disease,familyhistory_father_eczema,familyhistory_father_other_allergic_disease,familyhistory_sibling_eczema,familyhistory_sibling_other_allergic_disease
0,CA-001-LS,AD,Urban,24.0,2.0,1.0,1.0,11.0,1.0,0.0,...,negative,South Africa,0.0,17.0,0.0,0.0,0.0,0.0,NaN,NaN
1,CA-002-YF,AD,Urban,14.0,1.0,1.0,1.0,8.0,1.0,0.0,...,No stool,South Africa,0.0,15.0,0.0,1.0,0.0,0.0,0.0,0.0
2,CA-003-US,AD,Urban,20.0,2.0,1.0,1.0,4.0,3.0,0.0,...,No stool,South Africa,0.0,17.0,1.0,1.0,0.0,0.0,NaN,NaN
3,CA-004-NK,AD,Urban,14.0,1.0,1.0,1.0,1.0,1.0,0.0,...,No stool,South Africa,1.0,15.0,0.0,0.0,1.0,0.0,NaN,NaN
4,CA-005-BD,AD,Urban,17.0,2.0,1.0,1.0,6.0,1.0,0.0,...,negative,South Africa,0.0,17.0,1.0,0.0,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,CO-149-LM,HC,Rural,13.0,2.0,1.0,1.0,3.0,1.0,0.0,...,No stool,South Africa,0.0,15.0,0.0,0.0,0.0,0.0,NaN,NaN
213,CO-150-LN,HC,Rural,26.0,1.0,1.0,1.0,NaN,NaN,0.0,...,No stool,South Africa,0.0,17.0,0.0,0.0,0.0,0.0,NaN,NaN
214,CO-151-AN,HC,Rural,23.0,1.0,1.0,1.0,7.0,2.0,0.0,...,No stool,South Africa,0.0,17.0,0.0,0.0,0.0,0.0,0.0,0.0
215,CO-152-AF,HC,Rural,26.0,1.0,1.0,1.0,9.0,2.0,0.0,...,No stool,South Africa,0.0,17.0,0.0,0.0,0.0,0.0,NaN,NaN


### 3.6 Fuel category variables (fuel_heating & fuel_cooking)

In [40]:
### fuel usage variables
data_renamed.loc[:,data_renamed.columns.str.contains('fuel', regex = True)]

,fuel_cooking,fuel_heating
0,1,3
1,1,1
2,1,1
3,1,1
4,1,1
...,...,...
212,2;4,4
213,1,3
214,2;4,4
215,1,3


In [41]:
df_cooking = data_renamed["fuel_cooking"].fillna('Missing').str.get_dummies(";")
df_cooking.columns = ['fuel_cooking_Electricity_Gas', 'fuel_cooking_Paraffin_Stove','fuel_cooking_Open_fires_inside', 'fuel_cooking_Open_fires_outside', 'fuel_cooking_nan']

### assign NaNs to the entire row if the foodreaction_nan has NaNs
idx = df_cooking[df_cooking.fuel_cooking_nan == 1].index
df_cooking.loc[idx,:] = np.NaN
df_cooking

,fuel_cooking_Electricity_Gas,fuel_cooking_Paraffin_Stove,fuel_cooking_Open_fires_inside,fuel_cooking_Open_fires_outside,fuel_cooking_nan
0,1.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0
2,1.0,0.0,0.0,0.0,0.0
3,1.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...
212,0.0,1.0,0.0,1.0,0.0
213,1.0,0.0,0.0,0.0,0.0
214,0.0,1.0,0.0,1.0,0.0
215,1.0,0.0,0.0,0.0,0.0


In [42]:
### replace 0 with NaN hence there should be a value other than 0 if it's not missing
data_renamed['fuel_heating'] = data_renamed['fuel_heating'].replace({'0':np.nan, 0:np.nan})
df_heating = data_renamed["fuel_heating"].fillna('Missing').str.get_dummies(";")
df_heating.columns = ['fuel_heating_Electricity', 'fuel_heating_Gas','fuel_cooking_heating_Paraffin', 'fuel_heating_Wood_Coal','fuel_heating_Other', 'fuel_heating_nan']

### assign NaNs to the entire row if the foodreaction_nan has NaNs
idx = df_heating[df_heating.fuel_heating_nan == 1].index
df_heating.loc[idx,:] = np.NaN
df_heating

,fuel_heating_Electricity,fuel_heating_Gas,fuel_cooking_heating_Paraffin,fuel_heating_Wood_Coal,fuel_heating_Other,fuel_heating_nan
0,0.0,0.0,1.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0
2,1.0,0.0,0.0,0.0,0.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...
212,0.0,0.0,0.0,1.0,0.0,0.0
213,0.0,0.0,1.0,0.0,0.0,0.0
214,0.0,0.0,0.0,1.0,0.0,0.0
215,0.0,0.0,1.0,0.0,0.0,0.0


In [43]:
### compare fuel variables before and after transformation column by column
df_fuel_comparison = pd.concat([data_renamed['fuel_cooking'], df_cooking.drop(columns = 'fuel_cooking_nan'),
                                data_renamed['fuel_heating'], df_heating.drop(columns = 'fuel_heating_nan')],
                               axis = 1)
df_fuel_comparison.to_csv(path_out + 'DataFuelTransformation.txt', na_rep = 'NaN',sep = '\t')
df_fuel_comparison

,fuel_cooking,fuel_cooking_Electricity_Gas,fuel_cooking_Paraffin_Stove,fuel_cooking_Open_fires_inside,fuel_cooking_Open_fires_outside,fuel_heating,fuel_heating_Electricity,fuel_heating_Gas,fuel_cooking_heating_Paraffin,fuel_heating_Wood_Coal,fuel_heating_Other
0,1,1.0,0.0,0.0,0.0,3,0.0,0.0,1.0,0.0,0.0
1,1,1.0,0.0,0.0,0.0,1,1.0,0.0,0.0,0.0,0.0
2,1,1.0,0.0,0.0,0.0,1,1.0,0.0,0.0,0.0,0.0
3,1,1.0,0.0,0.0,0.0,1,1.0,0.0,0.0,0.0,0.0
4,1,1.0,0.0,0.0,0.0,1,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
212,2;4,0.0,1.0,0.0,1.0,4,0.0,0.0,0.0,1.0,0.0
213,1,1.0,0.0,0.0,0.0,3,0.0,0.0,1.0,0.0,0.0
214,2;4,0.0,1.0,0.0,1.0,4,0.0,0.0,0.0,1.0,0.0
215,1,1.0,0.0,0.0,0.0,3,0.0,0.0,1.0,0.0,0.0


In [44]:
### remove untrasformed fuel variables and append the new ones
df_fuel_transformed = pd.concat([df_cooking.drop(columns = 'fuel_cooking_nan'), df_heating.drop(columns = 'fuel_heating_nan')], axis = 1)
data_renamed.drop(labels = data_renamed.filter(regex ='fuel').columns, axis = 1, inplace = True)
data_renamed = pd.concat([data_renamed, df_fuel_transformed], axis = 1)
data_renamed

,PID,diagnosis,location,enrolement_age,gender,vaccination_status,paracetamol_exposure,paracetamol_exposure_age,paracetamol_exposure_time,childhood_inf_mumps,...,familyhistory_sibling_other_allergic_disease,fuel_cooking_Electricity_Gas,fuel_cooking_Paraffin_Stove,fuel_cooking_Open_fires_inside,fuel_cooking_Open_fires_outside,fuel_heating_Electricity,fuel_heating_Gas,fuel_cooking_heating_Paraffin,fuel_heating_Wood_Coal,fuel_heating_Other
0,CA-001-LS,AD,Urban,24.0,2.0,1.0,1.0,11.0,1.0,0.0,...,NaN,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,CA-002-YF,AD,Urban,14.0,1.0,1.0,1.0,8.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,CA-003-US,AD,Urban,20.0,2.0,1.0,1.0,4.0,3.0,0.0,...,NaN,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,CA-004-NK,AD,Urban,14.0,1.0,1.0,1.0,1.0,1.0,0.0,...,NaN,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,CA-005-BD,AD,Urban,17.0,2.0,1.0,1.0,6.0,1.0,0.0,...,NaN,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,CO-149-LM,HC,Rural,13.0,2.0,1.0,1.0,3.0,1.0,0.0,...,NaN,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
213,CO-150-LN,HC,Rural,26.0,1.0,1.0,1.0,NaN,NaN,0.0,...,NaN,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
214,CO-151-AN,HC,Rural,23.0,1.0,1.0,1.0,7.0,2.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
215,CO-152-AF,HC,Rural,26.0,1.0,1.0,1.0,9.0,2.0,0.0,...,NaN,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


### 3.7 Blood counts features

In [45]:
### blood counts transformation
df_blood = data_renamed.loc[:, data_renamed.columns.str.contains('blood_count')]

### function transforms every non numeric value into NaN
def isnumber(x):
    try:
        float(x)
        return True
    except:
        return False

### apply the function
df_blood = df_blood[df_blood.applymap(isnumber)]

### add a pseudocount to all the values in the dataframe (0.1)
### NB! I am not sure which value would be better here
### read more about log transformation and constant value addition
### this way it's possible to log transform the data
df_blood = df_blood.astype('float64')
df_blood = df_blood + 0.1

### log transform
df_blood_log = np.log(df_blood)
df_blood_log.columns = ['log_blood_counts_wcc', 'log_blood_counts_hb', 'log_blood_counts_plts',
                        'log_blood_counts_neutrophils', 'log_blood_counts_monocytes',
                        'log_blood_counts_lymphocytes', 'log_blood_counts_eosinophils', ]
df_blood_log.head(50)

,log_blood_counts_wcc,log_blood_counts_hb,log_blood_counts_plts,log_blood_counts_neutrophils,log_blood_counts_monocytes,log_blood_counts_lymphocytes,log_blood_counts_eosinophils
0,2.380472,2.442347,6.165628,1.229641,-0.162519,1.477049,0.900161
1,2.521721,2.433613,6.180224,1.269761,0.292670,1.948763,-0.174353
2,2.078191,2.388763,4.812997,0.837248,-0.713350,1.526056,-0.210721
3,2.559550,2.572612,5.673667,0.982078,-0.139262,1.968510,0.824175
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2.170196,2.388763,5.808443,1.169381,-1.021651,1.581038,-0.634878
6,2.332144,2.549445,5.886382,1.686399,-0.342490,1.353255,-0.494296
7,1.985131,2.424803,5.802420,1.342865,-0.198451,0.932164,-1.427116
8,2.578701,2.415914,NaN,1.358409,0.019803,2.089392,-1.021651
9,2.113843,2.388763,6.122712,1.508512,-1.049822,1.214913,-1.049822


In [46]:
### compare the untransformed and transformed blood variables
df_blood_comparison = pd.concat([data_renamed.loc[:, data_renamed.columns.str.contains('blood_count')], df_blood_log], axis = 1)
df_blood_comparison.to_csv(path_out + 'DataBloodLogTransformation.txt', na_rep = 'NaN',sep = '\t')
df_blood_comparison

,blood_count_wcc,blood_count_hb,blood_count_plts,blood_count_neutrophils,blood_count_monocytes,blood_count_lymphocytes,blood_count_eosinophils,log_blood_counts_wcc,log_blood_counts_hb,log_blood_counts_plts,log_blood_counts_neutrophils,log_blood_counts_monocytes,log_blood_counts_lymphocytes,log_blood_counts_eosinophils
0,10.71,11.4,476,3.32,0.75,4.28,2.36,2.380472,2.442347,6.165628,1.229641,-0.162519,1.477049,0.900161
1,12.35,11.3,483,3.46,1.24,6.92,0.74,2.521721,2.433613,6.180224,1.269761,0.292670,1.948763,-0.174353
2,7.89,10.8,123,2.21,0.39,4.50,0.71,2.078191,2.388763,4.812997,0.837248,-0.713350,1.526056,-0.210721
3,12.83,13.0,291,2.57,0.77,7.06,2.18,2.559550,2.572612,5.673667,0.982078,-0.139262,1.968510,0.824175
4,clotted,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,9.09,10.0,343,1.81,1.08,5.42,0.76,2.218116,2.312535,5.838022,0.647103,0.165514,1.708378,-0.150823
213,6.82,10.4,222,1.74,0.46,3.96,0.64,1.934416,2.351375,5.403128,0.609766,-0.579818,1.401183,-0.301105
214,10.72,11.8,342,2.24,1.26,6.84,0.34,2.381396,2.476538,5.835103,0.850151,0.307485,1.937302,-0.820981
215,10.25,10.5,257,3.49,0.96,5.02,0.72,2.336987,2.360854,5.549465,1.278152,0.058269,1.633154,-0.198451


In [47]:
### add the transformed blood counts to the final dataframe
data_renamed.drop(labels = data_renamed.filter(regex ='blood_count').columns, axis = 1, inplace = True)
data_renamed = pd.concat([data_renamed, df_blood_log], axis = 1)
data_renamed

,PID,diagnosis,location,enrolement_age,gender,vaccination_status,paracetamol_exposure,paracetamol_exposure_age,paracetamol_exposure_time,childhood_inf_mumps,...,fuel_cooking_heating_Paraffin,fuel_heating_Wood_Coal,fuel_heating_Other,log_blood_counts_wcc,log_blood_counts_hb,log_blood_counts_plts,log_blood_counts_neutrophils,log_blood_counts_monocytes,log_blood_counts_lymphocytes,log_blood_counts_eosinophils
0,CA-001-LS,AD,Urban,24.0,2.0,1.0,1.0,11.0,1.0,0.0,...,1.0,0.0,0.0,2.380472,2.442347,6.165628,1.229641,-0.162519,1.477049,0.900161
1,CA-002-YF,AD,Urban,14.0,1.0,1.0,1.0,8.0,1.0,0.0,...,0.0,0.0,0.0,2.521721,2.433613,6.180224,1.269761,0.292670,1.948763,-0.174353
2,CA-003-US,AD,Urban,20.0,2.0,1.0,1.0,4.0,3.0,0.0,...,0.0,0.0,0.0,2.078191,2.388763,4.812997,0.837248,-0.713350,1.526056,-0.210721
3,CA-004-NK,AD,Urban,14.0,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,2.559550,2.572612,5.673667,0.982078,-0.139262,1.968510,0.824175
4,CA-005-BD,AD,Urban,17.0,2.0,1.0,1.0,6.0,1.0,0.0,...,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,CO-149-LM,HC,Rural,13.0,2.0,1.0,1.0,3.0,1.0,0.0,...,0.0,1.0,0.0,2.218116,2.312535,5.838022,0.647103,0.165514,1.708378,-0.150823
213,CO-150-LN,HC,Rural,26.0,1.0,1.0,1.0,NaN,NaN,0.0,...,1.0,0.0,0.0,1.934416,2.351375,5.403128,0.609766,-0.579818,1.401183,-0.301105
214,CO-151-AN,HC,Rural,23.0,1.0,1.0,1.0,7.0,2.0,0.0,...,0.0,1.0,0.0,2.381396,2.476538,5.835103,0.850151,0.307485,1.937302,-0.820981
215,CO-152-AF,HC,Rural,26.0,1.0,1.0,1.0,9.0,2.0,0.0,...,1.0,0.0,0.0,2.336987,2.360854,5.549465,1.278152,0.058269,1.633154,-0.198451


### 3.8 Other transformations
- diagnosis_location column
- RNAseq column

In [48]:
### add column diagnosis location
data_renamed['diagnosis_location'] = data_renamed['diagnosis'] + '_' + data_renamed['location']
data_renamed

,PID,diagnosis,location,enrolement_age,gender,vaccination_status,paracetamol_exposure,paracetamol_exposure_age,paracetamol_exposure_time,childhood_inf_mumps,...,fuel_heating_Wood_Coal,fuel_heating_Other,log_blood_counts_wcc,log_blood_counts_hb,log_blood_counts_plts,log_blood_counts_neutrophils,log_blood_counts_monocytes,log_blood_counts_lymphocytes,log_blood_counts_eosinophils,diagnosis_location
0,CA-001-LS,AD,Urban,24.0,2.0,1.0,1.0,11.0,1.0,0.0,...,0.0,0.0,2.380472,2.442347,6.165628,1.229641,-0.162519,1.477049,0.900161,AD_Urban
1,CA-002-YF,AD,Urban,14.0,1.0,1.0,1.0,8.0,1.0,0.0,...,0.0,0.0,2.521721,2.433613,6.180224,1.269761,0.292670,1.948763,-0.174353,AD_Urban
2,CA-003-US,AD,Urban,20.0,2.0,1.0,1.0,4.0,3.0,0.0,...,0.0,0.0,2.078191,2.388763,4.812997,0.837248,-0.713350,1.526056,-0.210721,AD_Urban
3,CA-004-NK,AD,Urban,14.0,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,2.559550,2.572612,5.673667,0.982078,-0.139262,1.968510,0.824175,AD_Urban
4,CA-005-BD,AD,Urban,17.0,2.0,1.0,1.0,6.0,1.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AD_Urban
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,CO-149-LM,HC,Rural,13.0,2.0,1.0,1.0,3.0,1.0,0.0,...,1.0,0.0,2.218116,2.312535,5.838022,0.647103,0.165514,1.708378,-0.150823,HC_Rural
213,CO-150-LN,HC,Rural,26.0,1.0,1.0,1.0,NaN,NaN,0.0,...,0.0,0.0,1.934416,2.351375,5.403128,0.609766,-0.579818,1.401183,-0.301105,HC_Rural
214,CO-151-AN,HC,Rural,23.0,1.0,1.0,1.0,7.0,2.0,0.0,...,1.0,0.0,2.381396,2.476538,5.835103,0.850151,0.307485,1.937302,-0.820981,HC_Rural
215,CO-152-AF,HC,Rural,26.0,1.0,1.0,1.0,9.0,2.0,0.0,...,0.0,0.0,2.336987,2.360854,5.549465,1.278152,0.058269,1.633154,-0.198451,HC_Rural


In [49]:
### append a column with RNAseq flag from the existing file
old_data = pd.read_csv('~\\Documents\\Projects\\sosall\\raw-data\\final_data.csv')
data_renamed = pd.merge(data_renamed, old_data[['PID','RNASeq']], on='PID')
data_renamed

,PID,diagnosis,location,enrolement_age,gender,vaccination_status,paracetamol_exposure,paracetamol_exposure_age,paracetamol_exposure_time,childhood_inf_mumps,...,fuel_heating_Other,log_blood_counts_wcc,log_blood_counts_hb,log_blood_counts_plts,log_blood_counts_neutrophils,log_blood_counts_monocytes,log_blood_counts_lymphocytes,log_blood_counts_eosinophils,diagnosis_location,RNASeq
0,CA-001-LS,AD,Urban,24.0,2.0,1.0,1.0,11.0,1.0,0.0,...,0.0,2.380472,2.442347,6.165628,1.229641,-0.162519,1.477049,0.900161,AD_Urban,no
1,CA-002-YF,AD,Urban,14.0,1.0,1.0,1.0,8.0,1.0,0.0,...,0.0,2.521721,2.433613,6.180224,1.269761,0.292670,1.948763,-0.174353,AD_Urban,yes
2,CA-003-US,AD,Urban,20.0,2.0,1.0,1.0,4.0,3.0,0.0,...,0.0,2.078191,2.388763,4.812997,0.837248,-0.713350,1.526056,-0.210721,AD_Urban,no
3,CA-004-NK,AD,Urban,14.0,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,2.559550,2.572612,5.673667,0.982078,-0.139262,1.968510,0.824175,AD_Urban,no
4,CA-005-BD,AD,Urban,17.0,2.0,1.0,1.0,6.0,1.0,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AD_Urban,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,CO-149-LM,HC,Rural,13.0,2.0,1.0,1.0,3.0,1.0,0.0,...,0.0,2.218116,2.312535,5.838022,0.647103,0.165514,1.708378,-0.150823,HC_Rural,yes
213,CO-150-LN,HC,Rural,26.0,1.0,1.0,1.0,NaN,NaN,0.0,...,0.0,1.934416,2.351375,5.403128,0.609766,-0.579818,1.401183,-0.301105,HC_Rural,yes
214,CO-151-AN,HC,Rural,23.0,1.0,1.0,1.0,7.0,2.0,0.0,...,0.0,2.381396,2.476538,5.835103,0.850151,0.307485,1.937302,-0.820981,HC_Rural,yes
215,CO-152-AF,HC,Rural,26.0,1.0,1.0,1.0,9.0,2.0,0.0,...,0.0,2.336987,2.360854,5.549465,1.278152,0.058269,1.633154,-0.198451,HC_Rural,yes


### 4. Final dataframe saving and comparison to other data iteration

In [50]:
### make a report of the final dataframe
### make a table with feature value counts, number of unique values and missing value counts
counts = []
feature_name = []
na_count = []
perc_nan = []
length = []
for column in data_renamed.columns:
    ### calculate the value counts in each column
    val_counts = data_renamed[column].value_counts().to_dict()
    ### calculate the nan percentage in every column
    nan_sum = data_renamed[column].isna().sum()
    percent_missing = data_renamed[column].isnull().sum() * 100 / len(data_renamed[column])
    ### calculate the number of unique values
    length.append(len(data_renamed[column].unique()))
    ### append the values to list to construct the dataframe
    counts.append(val_counts)
    feature_name.append(column)
    na_count.append(nan_sum)
    perc_nan.append(percent_missing)

### construct dataframe
### for some reason counts NaN as a uniqe value but doesn't show it in the dictionary
df_report = pd.DataFrame(list(zip(feature_name, counts, length, na_count, perc_nan)),
                         columns = ['variable','value_counts','unique_values','nan_counts','nan_percentage'])
df_report.to_csv(path_out + 'feature_frequencies_report_data_modified.txt', sep = '\t')

In [51]:
df_report

,variable,value_counts,unique_values,nan_counts,nan_percentage
0,PID,"{'CA-001-LS': 1, 'CO-035-SM': 1, 'CO-024-KM': ...",217,0,0.000000
1,diagnosis,"{'AD': 116, 'HC': 101}",2,0,0.000000
2,location,"{'Rural': 112, 'Urban': 105}",2,0,0.000000
3,enrolement_age,"{17.0: 14, 14.0: 13, 12.0: 12, 19.0: 11, 23.0:...",30,0,0.000000
4,gender,"{1.0: 120, 2.0: 96}",3,1,0.460829
...,...,...,...,...,...
117,log_blood_counts_monocytes,"{-0.37106368139083207: 6, -0.35667494393873245...",115,43,19.815668
118,log_blood_counts_lymphocytes,"{1.3737155789130306: 3, 1.5260563034950492: 3,...",148,44,20.276498
119,log_blood_counts_eosinophils,"{-2.3025850929940455: 8, -1.07880966137193: 5,...",104,41,18.894009
120,diagnosis_location,"{'AD_Rural': 60, 'AD_Urban': 56, 'HC_Rural': 5...",4,0,0.000000


In [53]:
### save the final dataframe
data_renamed.to_csv(path_out + 'DataTransformed.txt', na_rep = 'NaN',sep = '\t')

In [54]:
### compare Marco's final to mine
data_marco = pd.read_csv('~\\Documents\\Projects\\sosall\\results\\preprocessing\\final_data_marco.csv')
data_marco.head(100)

,PID,diagnosis,location,enrolement_age,gender,vaccination_status,paracetamol_exposure,paracetamol_exposure_age,paracetamol_exposure_time,childhood_inf_mumps,...,fuel_cooking_Paraffin Stove,fuel_cooking_Electricity_Gas,fuel_cooking_Open fires in the house,fuel_cooking_Open fires outside the house,fuel_heating_Other,fuel_heating_Gas,fuel_heating_Electricity,fuel_heating_Kerosene_Paraffin,fuel_heating_Wood_coal,RNASeq
0,CA-001-LS,AD,Urban,24.0,female,complete,yes,11.0,1-10,no,...,no,yes,no,no,no,no,no,yes,no,no
1,CA-002-YF,AD,Urban,14.0,male,complete,yes,8.0,1-10,no,...,no,yes,no,no,no,no,yes,no,no,yes
2,CA-003-US,AD,Urban,20.0,female,complete,yes,4.0,>20,no,...,no,yes,no,no,no,no,yes,no,no,no
3,CA-004-NK,AD,Urban,14.0,male,complete,yes,1.0,1-10,no,...,no,yes,no,no,no,no,yes,no,no,no
4,CA-005-BD,AD,Urban,17.0,female,complete,yes,6.0,1-10,no,...,no,yes,no,no,no,no,yes,no,no,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,CA-140-EM,AD,Rural,32.0,male,complete,yes,2.0,1-10,no,...,no,yes,no,no,no,no,no,yes,no,yes
96,CA-141-TC,AD,Rural,15.0,female,complete,yes,3.0,1-10,no,...,no,yes,no,no,no,no,no,yes,no,yes
97,CA-142-AM,AD,Rural,21.0,female,complete,yes,3.0,10-20,no,...,no,yes,no,yes,no,no,no,yes,no,yes
98,CA-143-ST,AD,Rural,18.0,female,complete,yes,2.0,1-10,no,...,no,yes,no,yes,no,no,no,yes,no,yes


In [55]:
data_renamed.loc[:, data_renamed.columns.str.contains('fuel')].columns

Index(['fuel_cooking_Electricity_Gas', 'fuel_cooking_Paraffin_Stove',
       'fuel_cooking_Open_fires_inside', 'fuel_cooking_Open_fires_outside',
       'fuel_heating_Electricity', 'fuel_heating_Gas',
       'fuel_cooking_heating_Paraffin', 'fuel_heating_Wood_Coal',
       'fuel_heating_Other'],
      dtype='object')

In [56]:
### rename the fuel variables in Marco's file to have the same name as mine
data_marco.loc[:, data_marco.columns.str.contains('fuel')].columns = ['fuel_cooking_Paraffin_Stove', 'fuel_cooking_Electricity_Gas','fuel_cooking_Open_fires_inside', 'fuel_cooking_Open_fires_outside',
                                                                      'fuel_heating_Other', 'fuel_heating_Gas','fuel_heating_Electricity', 'fuel_heating_Paraffin', 'fuel_heating_Wood_Coal']
data_marco

,PID,diagnosis,location,enrolement_age,gender,vaccination_status,paracetamol_exposure,paracetamol_exposure_age,paracetamol_exposure_time,childhood_inf_mumps,...,fuel_cooking_Paraffin Stove,fuel_cooking_Electricity_Gas,fuel_cooking_Open fires in the house,fuel_cooking_Open fires outside the house,fuel_heating_Other,fuel_heating_Gas,fuel_heating_Electricity,fuel_heating_Kerosene_Paraffin,fuel_heating_Wood_coal,RNASeq
0,CA-001-LS,AD,Urban,24.0,female,complete,yes,11.0,1-10,no,...,no,yes,no,no,no,no,no,yes,no,no
1,CA-002-YF,AD,Urban,14.0,male,complete,yes,8.0,1-10,no,...,no,yes,no,no,no,no,yes,no,no,yes
2,CA-003-US,AD,Urban,20.0,female,complete,yes,4.0,>20,no,...,no,yes,no,no,no,no,yes,no,no,no
3,CA-004-NK,AD,Urban,14.0,male,complete,yes,1.0,1-10,no,...,no,yes,no,no,no,no,yes,no,no,no
4,CA-005-BD,AD,Urban,17.0,female,complete,yes,6.0,1-10,no,...,no,yes,no,no,no,no,yes,no,no,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
240,40010056,NaN,NaN,NaN,NaN,NaN,yes,10.0,1-10,no,...,no,yes,no,no,no,yes,no,no,no,no
241,40010052,NaN,NaN,NaN,NaN,NaN,yes,0.0,1-10,no,...,no,yes,no,no,no,yes,no,no,no,no
242,40010057,NaN,NaN,NaN,NaN,NaN,no,0.0,NaN,no,...,no,yes,no,no,no,no,no,yes,no,no
243,40010054,NaN,NaN,NaN,NaN,NaN,yes,0.0,>20,no,...,no,yes,no,no,no,no,no,yes,no,no


In [57]:
### compare datasets
df_all = pd.concat([data_marco, data_renamed],
                   axis='columns', keys=['marco', 'damir'])
df_all = df_all.swaplevel(axis='columns')[data_marco.columns[0:]]

### highlight the differences in excel file
def highlight_diff(data, color='yellow'):
    attr = 'background-color: {}'.format(color)
    other = data.xs('marco', axis='columns', level=-1)
    return pd.DataFrame(np.where(data.ne(other, level=0), attr, ''),
                        index=data.index, columns=data.columns)

df_high = df_all.style.apply(highlight_diff, axis=None)

df_high.to_excel(path_out + 'CompareOldNew.xlsx', na_rep = 'NaN')